# MINT Empathy Tactic Tagger Demo

**Tag supporter utterances with empathy tactics using fine-tuned LoRA adapters.**

This notebook runs the full MINT tactic tagging pipeline on your own conversation data. It uses 10 LoRA adapters (one per tactic) on top of Llama 3.1 8B Instruct to classify each supporter sentence.

**Requirements:**
- Google Colab with a **T4 GPU** (or better). Go to Runtime > Change runtime type > GPU.
- You must have **accepted Meta's Llama 3.1 license** on HuggingFace: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
- You must be logged into HuggingFace (the setup cell will prompt you).

**Tactics detected (10):** Information, Assistance, Advice, Validation, Emotional Expression, Paraphrasing, Self-Disclosure, Questioning, Reappraisal, Empowerment

In [ ]:
#@title 1. Setup: Install dependencies & check GPU
import subprocess, sys

# Install packages
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "vllm", "transformers", "huggingface_hub", "nltk", "accelerate"])

# Check GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU available: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    raise RuntimeError(
        "No GPU detected! Go to Runtime > Change runtime type > GPU (T4 recommended)."
    )

# Login to HuggingFace (needed for gated Llama model)
from huggingface_hub import login
login()

# Download NLTK sentence tokenizer
import nltk
nltk.download("punkt_tab", quiet=True)

print("Setup complete.")

## 2. Download LoRA Adapters

This downloads the 10 tactic-specific LoRA adapters from [`hongli-zhan/empathy-tactic-taggers-llama3.1-8b`](https://huggingface.co/hongli-zhan/empathy-tactic-taggers-llama3.1-8b).

The adapters are small (~50 MB each) and will be cached locally.

In [ ]:
#@title 2. Download LoRA adapters from HuggingFace
from huggingface_hub import snapshot_download
import os

ADAPTER_REPO = "hongli-zhan/empathy-tactic-taggers-llama3.1-8b"
ADAPTER_DIR = "./lora_adapters"

TACTICS = [
    "information", "assistance", "advice", "validation", "emotional_expression",
    "paraphrasing", "self_disclosure", "questioning", "reappraisal", "empowerment"
]

# Download the full repo (contains all adapter subfolders)
local_path = snapshot_download(repo_id=ADAPTER_REPO, local_dir=ADAPTER_DIR)

# Verify adapters exist
found = []
for tactic in TACTICS:
    adapter_path = os.path.join(ADAPTER_DIR, f"Llama-3.1-8B-Instruct-tagger-{tactic}")
    if os.path.isdir(adapter_path):
        found.append(tactic)
    else:
        print(f"  Warning: adapter not found for '{tactic}'")

print(f"Downloaded {len(found)}/{len(TACTICS)} adapters to {ADAPTER_DIR}/")

## 3. Your Conversation

Edit the cell below with your own conversation. Each turn should have:
- `"role"`: either `"seeker"` (the person seeking support) or `"supporter"` (the empathic responder)
- `"content"`: the text of that turn

Only **supporter** turns will be tagged with empathy tactics.

In [ ]:
#@title 3. Paste your conversation here
# Edit this list with your own conversation.
# Only "supporter" turns will be analyzed for empathy tactics.

conversation = [
    {
        "role": "seeker",
        "content": "I've been feeling really overwhelmed at work lately. My manager keeps piling on tasks and I don't know how to say no."
    },
    {
        "role": "supporter",
        "content": "That sounds incredibly stressful, and it makes total sense that you're feeling overwhelmed. Setting boundaries at work is really hard, especially when you want to be seen as reliable. Have you thought about scheduling a one-on-one with your manager to talk about your workload? Sometimes just being honest about your capacity can make a big difference."
    },
    {
        "role": "seeker",
        "content": "I'm scared they'll think I'm not capable or that I'm complaining."
    },
    {
        "role": "supporter",
        "content": "I completely understand that fear. I've been in a similar situation before where I worried about how I'd be perceived. But advocating for yourself isn't complaining. Most managers actually appreciate when someone is upfront about bandwidth because it prevents burnout and mistakes down the line. You clearly care about doing good work, and that's exactly why this conversation is worth having."
    },
]

print(f"Conversation loaded: {len(conversation)} turns")
print(f"Supporter turns to tag: {sum(1 for t in conversation if t['role'] == 'supporter')}")

## 4. Run the Tactic Tagger

This cell loads the base model with vLLM, applies each LoRA adapter in sequence, and tags every supporter sentence. This takes 2-5 minutes on a T4 GPU.

In [ ]:
#@title 4. Run tagger on your conversation
import re
import os
from nltk.tokenize import sent_tokenize
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from transformers import AutoTokenizer

# --- Configuration ---
BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
ADAPTER_DIR = "./lora_adapters"
TACTICS = [
    "information", "assistance", "advice", "validation", "emotional_expression",
    "paraphrasing", "self_disclosure", "questioning", "reappraisal", "empowerment"
]

# --- Prompt templates (embedded from tactic_tagger/prompts/) ---
# Each prompt uses {Full_Response} and {Sentence} placeholders.
PROMPTS = {}
for tactic in TACTICS:
    prompt_path = os.path.join(ADAPTER_DIR, "..", "tactic_tagger", "prompts", f"{tactic}.txt")
    # Try local prompts dir first, fall back to reading from the adapter repo
    local_prompt = os.path.join(ADAPTER_DIR, "prompts", f"{tactic}.txt")
    if os.path.exists(local_prompt):
        with open(local_prompt) as f:
            PROMPTS[tactic] = f.read()
    else:
        # Prompts are also bundled in the adapter repo
        # If not found, we use a generic template
        PROMPTS[tactic] = None

# If prompts weren't in the adapter repo, download them from the main repo
if any(v is None for v in PROMPTS.values()):
    from huggingface_hub import hf_hub_download
    for tactic in TACTICS:
        if PROMPTS[tactic] is None:
            try:
                path = hf_hub_download(
                    repo_id="hongli-zhan/mint-empathy",
                    filename=f"tactic_tagger/prompts/{tactic}.txt",
                    repo_type="model"
                )
                with open(path) as f:
                    PROMPTS[tactic] = f.read()
            except Exception:
                # Fallback: generic prompt template
                cap_name = tactic.replace('_', ' ').title()
                PROMPTS[tactic] = f'''### Instruction:\n\n1. You will be provided with a full empathic response for context and a single sentence extracted from it. Your task is to determine whether the given sentence contains "{cap_name}".\n\n2. Read the sentence and provide a rating of 0 or 1, with 0 signifying that "{cap_name}" is not present and 1 signifying that it is present. Your response should be in the following format: <score>[]</score>\n\n### Input:\n- Context (Full Empathic Response):\n{{Full_Response}}\n\n- Sentence to Evaluate:\n{{Sentence}}\n\n### Response:\n'''

# --- Build system prompts ---
def get_system_prompt(tactic):
    cap_name = tactic.replace('_', ' ').title()
    return (
        f'You are a Fair Tagger Assistant, responsible for providing precise, '
        f'objective tagging based on predefined criteria. Your task is to assess '
        f'whether a given sentence contains "{cap_name}", ensuring consistency '
        f'and adherence to strict tagging guidelines.'
    )

# --- Extract supporter sentences ---
sentences_to_tag = []
for turn_idx, turn in enumerate(conversation):
    if turn["role"] == "supporter":
        full_response = turn["content"]
        sents = sent_tokenize(full_response)
        for sent_idx, sent in enumerate(sents):
            sentences_to_tag.append({
                "turn_idx": turn_idx,
                "sent_idx": sent_idx,
                "full_response": full_response,
                "sentence": sent,
            })

print(f"Supporter sentences to tag: {len(sentences_to_tag)}")
print(f"Total inference calls: {len(sentences_to_tag)} sentences x {len(TACTICS)} tactics = {len(sentences_to_tag) * len(TACTICS)}")
print()

# --- Load model ---
print("Loading model (this may take 1-2 minutes)...")
llm = LLM(
    model=BASE_MODEL,
    enable_lora=True,
    max_lora_rank=64,
    dtype="bfloat16",
    gpu_memory_utilization=0.85,
    max_model_len=4096,
    tensor_parallel_size=1,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print("Model loaded.")

sampling_params = SamplingParams(
    temperature=0.1,
    max_tokens=512,
    top_p=0.9,
    top_k=50,
)

# --- Tag each tactic ---
results = {}  # (turn_idx, sent_idx) -> list of tactic names

for tactic_idx, tactic in enumerate(TACTICS):
    adapter_path = os.path.join(ADAPTER_DIR, f"Llama-3.1-8B-Instruct-tagger-{tactic}")
    if not os.path.isdir(adapter_path):
        print(f"  Skipping {tactic} (adapter not found)")
        continue

    lora_req = LoRARequest(tactic, tactic_idx + 1, adapter_path)
    prompt_template = PROMPTS[tactic]
    system_prompt = get_system_prompt(tactic)

    # Build prompts for all sentences
    prompts = []
    for item in sentences_to_tag:
        user_msg = prompt_template.replace("{Full_Response}", item["full_response"]).replace("{Sentence}", item["sentence"])
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg},
        ]
        prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(prompt_str)

    # Batch inference
    outputs = llm.generate(prompts, sampling_params=sampling_params, lora_request=lora_req)

    # Parse results
    tagged_count = 0
    for item, output in zip(sentences_to_tag, outputs):
        text = output.outputs[0].text
        match = re.search(r'<score>(\d+)</score>', text)
        score = int(match.group(1)) if match else 0
        if score == 1:
            key = (item["turn_idx"], item["sent_idx"])
            if key not in results:
                results[key] = []
            results[key].append(tactic)
            tagged_count += 1

    print(f"  [{tactic_idx+1}/{len(TACTICS)}] {tactic}: {tagged_count}/{len(sentences_to_tag)} sentences tagged")

print(f"\nTagging complete! {len(results)} sentences have at least one tactic.")

## 5. Results

Below you'll see each supporter turn with its sentences highlighted by detected empathy tactics.

In [ ]:
#@title 5. Display results with colored tactic badges
from IPython.display import display, HTML
from nltk.tokenize import sent_tokenize

# Color palette for tactics
TACTIC_COLORS = {
    "information": "#4A90D9",
    "assistance": "#7B68EE",
    "advice": "#E67E22",
    "validation": "#27AE60",
    "emotional_expression": "#E74C3C",
    "paraphrasing": "#16A085",
    "self_disclosure": "#8E44AD",
    "questioning": "#F39C12",
    "reappraisal": "#2980B9",
    "empowerment": "#D35400",
}

def make_badge(tactic):
    color = TACTIC_COLORS.get(tactic, "#95A5A6")
    label = tactic.replace("_", " ").title()
    return (
        f'<span style="display:inline-block; background-color:{color}; color:white; '
        f'padding:2px 8px; border-radius:12px; font-size:12px; font-weight:bold; '
        f'margin:2px 3px;">{label}</span>'
    )

# Build HTML output
html_parts = ['<div style="font-family: sans-serif; max-width: 800px;">']

supporter_idx = 0
for turn_idx, turn in enumerate(conversation):
    if turn["role"] == "seeker":
        html_parts.append(
            f'<div style="margin:16px 0; padding:12px 16px; '
            f'background:#f0f0f0; border-radius:12px; border-left:4px solid #bbb;">'
            f'<strong style="color:#555;">Seeker:</strong><br>'
            f'{turn["content"]}</div>'
        )
    else:
        html_parts.append(
            f'<div style="margin:16px 0; padding:12px 16px; '
            f'background:#e8f4fd; border-radius:12px; border-left:4px solid #4A90D9;">'
            f'<strong style="color:#2c3e50;">Supporter:</strong>'
        )
        sents = sent_tokenize(turn["content"])
        for sent_idx, sent in enumerate(sents):
            key = (turn_idx, sent_idx)
            tactics = results.get(key, [])
            badges = " ".join(make_badge(t) for t in tactics)
            bg = "#fff3cd" if tactics else "transparent"
            html_parts.append(
                f'<div style="margin:8px 0; padding:8px; border-radius:6px; background:{bg};">'
                f'<span style="color:#333;">{sent}</span><br>'
                f'{badges if badges else "<em style=\"color:#999; font-size:12px;\">(no tactics detected)</em>"}'
                f'</div>'
            )
        html_parts.append('</div>')

html_parts.append('</div>')

# Legend
html_parts.append('<div style="margin-top:24px; padding:12px; background:#f9f9f9; border-radius:8px;">')
html_parts.append('<strong>Legend:</strong><br>')
for tactic, color in TACTIC_COLORS.items():
    html_parts.append(make_badge(tactic))
html_parts.append('</div>')

display(HTML("\n".join(html_parts)))